In [0]:
%run "../00.common/01.environment-config.py"

In [0]:
%run "../00.common/02.bronze-helpers.py"

In [0]:
# COMMAND ----------
 
# Load environment configuration
 
# MAGIC %run "../00_common/01_environment_config"
 
 
# COMMAND ----------
 
# Load Bronze helper functions
 
# MAGIC %run "../00_common/02_bronze_helpers"
 
 
# COMMAND ----------
 
# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================
 
source_file = f"{s3_root_path}/landing/circuits.csv"
 
target_path = f"{s3_root_path}/Bronze/Nisarga/circuits"
 
print("Source File :", source_file)
print("Target Path :", target_path)
 
 
# COMMAND ----------
 
# ============================================================
# DEFINE SCHEMA
# ============================================================
 
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)
 
circuits_schema = StructType([
    StructField("circuitId", StringType(), True),
    StructField("url", StringType(), True),
    StructField("circuitName", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("long", DoubleType(), True),
    StructField("locality", StringType(), True),
    StructField("country", StringType(), True)
])
 
 
# COMMAND ----------
 
# ============================================================
# READ CIRCUITS DATA FROM S3 LANDING
# ============================================================
 
circuits_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("mode", "FAILFAST")
    .schema(circuits_schema)
    .load(source_file)
)
 
display(circuits_df)
 
 
# COMMAND ----------
 
# Check schema
 
circuits_df.printSchema()
 
 
# COMMAND ----------
 
# ============================================================
# ADD INGESTION METADATA
# ============================================================
 
circuits_final_df = add_ingestion_metadata(circuits_df)
 
display(circuits_final_df)
 
 
# COMMAND ----------
 
# ============================================================
# WRITE DATA TO S3 BRONZE LAYER IN DELTA FORMAT
# ============================================================
 
(
    circuits_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(target_path)
)
 
 
# COMMAND ----------
 
# ============================================================
# VALIDATE BRONZE DATA
# ============================================================
 
bronze_circuits_df = (
    spark.read
    .format("delta")
    .load(target_path)
)
 
display(bronze_circuits_df)
 
 
# COMMAND ----------
 
# Validate record count
 
print("Total Bronze Records:", bronze_circuits_df.count())
 
print("Bronze Circuits data successfully written to:")
print(target_path)